In [1]:
import sqlite3
import pandas as pd
from tqdm import tqdm
import numpy as np
import copy
import os
import configparser
import json
import time

In [2]:
pd.set_option('display.max_colwidth', None)

## Vastustega df

In [6]:
df = pd.read_csv("../gpt_output/n80_examples_large_v1_gpt_v1_10K_b12_v1.csv", encoding="utf-8", sep="|")

In [7]:
df

,head_id,form,lemma,verb,verb_compound,morph_case,sentence_id,sentence,timex_tag,ekilex_tag,ner_tag,classification,explanation
0,12979894,autodesse,auto,müüma,NaN,ill,8089054,“ Nad tassivad raskeid mitmekiloseid pakke ja müüvad lehti otse keset liiklust autodesse .,NaN,NaN,NaN,yes,"The term 'autodesse' refers to cars, which are considered a physical location, so it is classified as yes."
1,18564119,põõsas,põõsas,kükitama,NaN,in,11594114,"Kõik need praegused tähtsad tegelased kükitasid põõsas ja ootasid aega , et istuda toolile , mille teised olid kätte võidelnud .",NaN,NaN,NaN,yes,NaN
2,6712022,vitriinides,vitriin,ilutsema,NaN,in,4170566,"Kunstnike fantaasia võtab silme eest kirjuks - vitriinides ilutsevad rohelised , sinised , kollased , lillelised , liblikamustriga jm.",NaN,NaN,NaN,yes,NaN
3,15715,MTVs,MTV,ringlema,NaN,in,9160,"Kolm Sahlene soolosinglit on ringelnud MTVs , jõudes MTV Nordic Top 5 hulka .",NaN,NaN,NaN,no,"The term 'MTVs' refers to a TV network brand and not a geographic place, so it is not classified as a location."
4,9552490,töökohta,töökoht,pöörduma,tagasi,adit,5950379,"Hamburgis tehtud uuringu põhjal selgus , et ligi 70 protsenti naistest pöörduks pärast esimese lapse sündi heameelega vanasse töökohta tagasi , tegelikkuses teeb seda aga vaid 58 protsenti värsketest emadest .",NaN,NaN,NaN,yes,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,10441673,Kumusse,kumu,hakkama,NaN,ill,6492455,"Näiteks tunnelit , mille kaudu Kumusse veoautod väärtuslike maalikoormatega sõitma hakkavad , polnud üldse alguses plaanis ehitadagi .",NaN,NaN,NaN,yes,NaN
9996,21044230,Texases,Texas,tulistama,NaN,in,13152394,"Cheney jahilembus pääses tänavu ka meediasse , kui ta Texases kogemata oma linnujahikaaslast tulistas .",NaN,location,LOC,yes,NaN
9997,26748522,Liitu,liit,pagema,NaN,adit,17445921,"Kui riik seda ei teeks , toimuks paari põlvkonna jooksul perifeerias , näiteks , Piirissaarel ja Ruhnu saarel , Alutaguse metsades ja mujal migratsioon , tühjenemine ning Euroopa Liit võiks teha 21. sajandi alguseks Eestile ettepaneku asustada nendele tühjenenud maadele Euroopa Liitu varju pagenud pagulasi kolmandast maailmast , mis tekitaks Eestile tõenäoliselt sama keerulise probleemi kui hetkel riigi ääremaade rahaline abistamine .",NaN,NaN,NaN,no,"The phrase 'Liitu' refers to 'European Union', which is an organization and not a location, hence classified as 'no'."
9998,1772660,meediasse,meedia,hakkama,NaN,ill,1114969,"Ka eesti meediasse on hakanud imbuma uudiseid ja usutlusi memeetikast - viimaste aastakümnetega esiletõusnud teadusharust , mis uurib ideede evolutsiooni ja mõjulepääsmist teadvuses ja kultuurides .",NaN,NaN,NaN,no,"The phrase 'meediasse' translates to 'into the media', which is a concept and not a specific location, hence classified as 'no'."


In [45]:
saving_columns = ["head_id", "form", "lemma", "verb", "verb_compound", "morph_case", "sentence_id", "sentence", "timex_tag", "ekilex_tag", "ner_tag"]

In [97]:
print(list(df["lemma"].unique()))

['auto', 'põõsas', 'vitriin', 'MTV', 'töökoht', 'peatus', 'Hiina', 'dzhungel', 'maneež', 'Kuressaare', 'That', 'ajakirjandus', 'haigla', 'Kaubanduskeskus', 'maja', 'USA', 'motopood', 'Kaasan', 'liit', 'Tartu', 'Kambodža', 'kaugõpe', 'jaamahoone', 'type9', 'Kenema', 'elutuba', 'play-offi', 'liikmesriik', 'transformaator', 'Pirita', 'pea', 'Nevada', 'erasfäär', 'Dubai', 'Panga', 'Invernessi', 'haldusõigus', 'sport', 'Kuuba', 'pori', 'püünis', 'Ameerika', 'koduküla', 'Korintos', 'kabiin', 'Riia', 'stuudio', 'korter', 'kool', 'Eesti', 'kraadiõpe', 'alevik', 'agent', 'hoidla', 'metsapiirkond', 'suund', 'tualett', 'kelder', 'väikeapteek', 'Slovakkia', 'igapäevaelu', 'Argentina', 'Räpina', 'rohukapp', 'boks', 'Jurmala', 'huul', 'köis', 'foto', 'Calgary', 'faas', 'maa', 'vallamaja', 'pirukas', 'kolle', 'hõbe', 'autoturg', 'Peking', 'laager', 'C-grupp', 'põhjaosa', 'London', 'rutiin', 'NLKP', 'kava', 'Nicaragua', 'Austraalia', 'kodu', 'firma', 'metallikool', 'Norra', 'kelleg', 'lennuk', 'lasteh

### object_loc

* füüsilised objektid: esikohapoodium, Kuu, varundusseade, sadul, pilv, põuetasku
* kui väljendatakse abstraktset nähtust, aga objekti geograafiline asukoht on ikka määratav (nt hirm käib luust läbi - tegelikult pole hirm luus, aga luu on ise ikkagi kindla asukohaga)


In [103]:
searchfor = ["seade", "poodium", "mänguasi", "pirukas", "tasku", "pilv", "karp", "masin", "lennuk", "käru", "vitriin", "sadul", "putka", "auto", "katel", "kott", "ketas", "päevik", "kasukas", "lehv", "medal", "uks", "ratas"]
undes = ["turg", "bussitasku", "autoinspektsioon", "vang", "masingam", "pesula", "Luksemburg", "Aluksne", "medalikolmik", "tehas", "salong", "äri", "firma", "keskus", "pood", "baas", "parkla", "hotell", "automaat", "autorikaitse", "avarii", "register", "teenindus", "autotee", "võidusõit", "esikuuks"]
obj1 = df[(df["lemma"].str.contains('|'.join(searchfor))) &  ~(df["lemma"].str.contains('|'.join(undes))) | df["form"].str.contains("silmadest") | df["form"].str.contains("näkku")]
obj1 = obj1.iloc[:100]
obj1 = obj1.sample(frac=1)
obj1 = obj1[saving_columns]
obj1

,head_id,form,lemma,verb,verb_compound,morph_case,sentence_id,sentence,timex_tag,ekilex_tag,ner_tag
2,6712022,vitriinides,vitriin,ilutsema,NaN,in,4170566,"Kunstnike fantaasia võtab silme eest kirjuks - vitriinides ilutsevad rohelised , sinised , kollased , lillelised , liblikamustriga jm.",NaN,NaN,NaN
4503,12695817,karpi,karp,sulgema,NaN,adit,7924683,"Nii et pärast kerget kohvikueinet tulime oma nõndanimetatud hotelli tagasi , maksime — peremehe rõõmuks , sest külalisi oli tal nii varasel aastaajal alles õige harva , oma toa eest järgmise hommikuni ja sulgesime end täies anonüümsuses sesse tüütult ja lohutavalt roosalillelise tapeediga karpi .",NaN,NaN,NaN
5308,1964719,autos,auto,olema,alles,in,1235891,Ka Jevgeni mobiiltelefon oli autos alles .,NaN,NaN,NaN
1197,2509163,lennukitelt,lennuk,kukkuma,alla,abl,1577268,Jäätunud veekamakad kukuvad lennukitelt alla ja maanduvad inimeste eluamajadele ning aedadesse .,NaN,NaN,NaN
4630,13436270,autosse,auto,investeerima,NaN,ill,8384613,Uude autosse investeerib Opel ligikaudu 300 miljonit Saksa marka .,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
2497,9245685,teokarbist,teokarp,voolama,välja,el,5763409,"Ka vetejumala jalgade juures olevast teokarbist voolab välja vesi , mis valgub mööda kaskaadi astmeid allapoole .",NaN,NaN,NaN
5088,4639953,mänguasjadesse,mänguasi,kaduma,NaN,ill,2893878,"Elusatesse mänguasjadesse kaob aja jooksul usk , kuid neid , kes veel keskeaski ei julge öösel pimedas üle toa käia , on meie seas hämmastavalt palju .",NaN,NaN,NaN
1982,22450863,mänguasjast,mänguasi,voolama,välja,el,14102418,"2 ) mänguasjas sisalduvad vedelikud ja gaasid ei saavutaks sellist temperatuuri ega rõhku , et nad voolaksid mänguasjast välja muul juhul , kui on vajalik mänguasja funktsioneerimiseks , ning võiksid tekitada põletuse või muu kehalise vigastuse ohtu .",NaN,NaN,NaN
3063,4426858,puidukoorimismasinasse,puidukoorimismasin,kukkuma,NaN,ill,2760280,Keilas asuva ettevõtte territooriumil hukkus eile hommikul puidukoorimismasinasse kukkunud noor mees .,NaN,NaN,NaN


In [104]:
obj1["lemma"].unique()

array(['vitriin', 'karp', 'auto', 'lennuk', 'lennukikandja', 'masin',
       'uks', 'CD-karp', 'tootmisseade', 'seade', 'autoaken', 'reisikott',
       'tagaratas', 'katel', 'kilekott', 'mootorlennuk', 'pirukas',
       'rahakott', 'kõvaketas', 'lehv', 'väikeveoauto', 'välisuks',
       'jopetasku', 'käekott', 'silm', 'rahatasku', 'sõiduauto',
       'autojuhiluba', 'pesumasin', 'autorool', 'metallkarp', 'sportauto',
       'putka', 'klassipäevik', 'jakitasku', 'kasukas', 'tasku', 'kott',
       'korjanduskarp', 'medal', 'ratas', 'elektriauto', 'miilitsaauto',
       'pilv', 'puituks', 'autokatus', 'nägu', 'lihakäru', 'muusikamasin',
       'laudakarp', 'keevaveekatel', 'lumepilv', 'küljetasku', 'juhiuks',
       'teokarp', 'mänguasi', 'puidukoorimismasin'], dtype=object)

In [105]:
obj1.to_csv("object_loc/object_loc_testset100.csv", sep="|", encoding="utf-8", index=False)

### org_loc
* organisatsioonid/kollektiivid: istun valitsuses, käin ülikoolis, hokitrennis, liigun võrgustikesse, lahkun töökohalt, vormelimaailm (koolid, trennid, lasteaiad)

In [88]:
searchfor = ["trenn", "laager", "valitsus", "töökoht", "kool", "liit", "nõukogu"]
undes = ["koolkond", "koolimaja", "koolikoridor", "koolihoone", "koolihoov", "koolisöökla", "keskkooliaste", "otsa-kool", "ülikoolilinn", "valitsuskvartal", "monoliitsus", "poliitik", "eliit"]
org1 = df[(df["lemma"].str.contains('|'.join(searchfor))) &  ~(df["lemma"].str.contains('|'.join(undes)))]
org1 = org1.iloc[:100]
org1 = org1.sample(frac=1)
org1 = org1[saving_columns]
org1

,head_id,form,lemma,verb,verb_compound,morph_case,sentence_id,sentence,timex_tag,ekilex_tag,ner_tag
3906,16081681,koolidesse,kool,lähetama,NaN,ill,10024651,"Siinkohal tasub Tartu Ülikooli rektoril kahe käe näppudel ära lugeda , kui mitu noort õpetajat ta läinud aastal oma kodumaa koolidesse lähetas .",NaN,NaN,NaN
3066,8792865,koonduslaagris,koonduslaager,laskma,maha,in,5484471,"Malloth lasi ühe juudi maha 1943. aastal Theresienstadti koonduslaagris , mis asus praeguse Tšehhi territooriumil .",NaN,location,NaN
1344,10083043,kooli,kool,kandma,NaN,adit,6276069,"Rekordimees Heiki Ojasild Kesklinna kooli IV klassist kannab iga päev kodunt kooli ja koolist koju 6,8 kilo kaaluvat ranitsat .",NaN,NaN,NaN
5673,27046437,nõukogusse,nõukogu,juhtuma,NaN,ill,17706403,"Olles olnud eelmise koosseisu ajal Põllumajanduse ja Maaelu Krediteerimise sihtasutuse nõukogu liige ja teades , et see ei allu ega ole allunud Riigikontrolli kontrollile , söandan ma väita , et juhul kui selle fondi nõukogusse juhtub ka niisuguseid krutskitega inimesi , siis on seal võrdlemisi suuri võimalusi pehmelt öeldes pahategudeks , mida kindlasti ei oleks sellisel juhul , kui tegemist oleks avalik-õigusliku juriidilise isikuga , mis seaduse järgi oleks pidanud toimima hakkama 1. juunist .",NaN,location,NaN
6506,3401265,usukoolides,usukool,baseeruma,NaN,in,2129872,"Saudi-Araabia salaluure Istakhbarat oli juba 1995. aasta lõpul otsustanud alustada raha andmist Talibanile , mis toona baseerus peamiselt Pakistani usukoolides .",NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
1444,15099134,koolides,kool,hoidma,kinni,in,9403082,"Niisiis tuleks alustada inglise , saksa ja prantsuse keele õpetajate massilisest ettevalmistamisest , luues ühtlasi muidugi tingimused , mis neid koolides kinni hoiaks ega sunniks otsima tasuvamat ja meeldivamat tööd näiteks tõlgina .",NaN,NaN,NaN
4236,13615892,kloostrikoolis,kloostrikool,õpetama,NaN,in,8494803,"Ta naases Vologdamaale ja õpetas pea pool aastat Zaozerski erakla kloostrikoolis ning valmistus preestriks saama , ent sunniti sealt lahkuma kui lahkhelide tekitaja ja ateist , kes muuseas propageeris Darwini õpetust .",NaN,location,NaN
903,12407471,laagritest,laager,pagema,NaN,el,7739072,Juhuste läbi jõudis Sondasse ka Vene laagritest pagenud ja valedokumente kasutanud isa .,NaN,NaN,NaN
845,5809650,Tehnikaülikoolis,Tehnikaülikool,tudeerima,NaN,in,3607649,Sügisest tudeerib tippujuja Tallinna Tehnikaülikoolis majandust .,NaN,NaN,ORG


In [89]:
org1["lemma"].unique()

array(['kool', 'koonduslaager', 'nõukogu', 'usukool', 'Tehnikaülikool',
       'laager', 'ülikool', 'põgenikelaager', 'linnavalitsus',
       'lumelaager', 'julgeolekunõukogu', 'omavalitsus', 'trenn',
       'teatriliit', 'merelaager', 'kommertskool', 'lõunalaager',
       'mäestikulaager', 'liit', 'septembrilaager', 'üldhariduskool',
       'puhastuslaager', 'aianduskool', 'kutsekool', 'vangilaager',
       'pedagoogikaülikool', 'keskkool', 'töökoht', 'algkool',
       'asjadevalitsus', 'erikool', 'kontsentratsioonilaager',
       'maavalitsus', 'suvelaager', 'metallikool', 'külakool',
       'spordikool', 'distsiplinaarlaager', 'valitsus', 'koondisetrenn',
       'põhikool', 'munitsipaalkool', 'kõrgkool', 'alpilaager',
       'kloostrikool'], dtype=object)

In [90]:
org1.to_csv("org_loc/org_loc_testset100.csv", sep="|", encoding="utf-8", index=False)

### event_loc

* kleidiproov, värbamine, haldusmenetlus, prostitutsiooniprotsess, suusatreening jne

In [51]:
searchfor = ["kleidiproov", "värbamine", "menetlus", "protsess", "treening"]
undes = []
ev1 = df[(df["lemma"].str.contains('|'.join(searchfor)))]
ev1

,head_id,form,lemma,verb,verb_compound,morph_case,sentence_id,sentence,timex_tag,ekilex_tag,ner_tag,classification,explanation
2687,3283344,menetlusse,menetlus,tulema,tagasi,adit,2058268,"Arvan , et siis tuleb ta õige ruttu parlamendi menetlusse tagasi , "" uskus Vilosius .",NaN,NaN,NaN,no,"The word 'menetlusse' refers to a process or procedure, not a location, which is why it is classified as 'no'."
6258,16001699,treeningrühmadesse,treeningrühm,ootama,NaN,ill,9975998,"Nädala kolmel esimesel päeval kell 15 ootab Jaanus Teppan laululava juures oma treeningrühmadesse uusi 8-12-aastasi poisse-tüdrukuid , tagasi ei saadeta ka pisut vanemaid suusahuvilisi lapsi .",NaN,alive,NaN,no,"The phrase 'treeningrühmadesse' refers to training groups, which are not physical locations but rather organized activities."
6970,970361,protsessist,protsess,pääsema,välja,el,611950,""" Aga Tallinna linn ei pääse sellest protsessist välja isegi mitte teoreetiliselt , "" kinnitas Lepik .",NaN,NaN,NaN,no,"The phrase 'protsessist' refers to a process and not a geographic location, so it is classified as 'no'."
8439,12412586,protsessides,protsess,ringlema,NaN,in,7742460,"Raske on veekogust välja püüda seal massiliselt kasvavaid taimekogumeid ( lemled , kardhein ) , aga sügise poole hakkavad nad lagunema , andes lisa biogeenidele , mis ringlevad veekogusisestes protsessides .",NaN,NaN,NaN,no,"'protsessides' refers to processes or activities, not a location."
8958,3341743,Kohtuprotsessides,kohtuprotsess,ringlema,NaN,in,2094039,Kohtuprotsessides ja politseipaberites ringleb neid vaid paarkümmend .,NaN,NaN,NaN,no,"The phrase 'Kohtuprotsessides' refers to legal proceedings or trials, which are events or activities, not a physical location, so it was classified as 'no'."
9713,20388279,aeroobikatreeningutesse,aeroobikatreening,suunama,NaN,ill,12745839,Mina olen suunanud naisi õige koormusega aeroobikatreeningutesse .,NaN,event,NaN,yes,NaN


In [91]:
len(ev1)

6

### per_loc

In [ ]:
# ema, kaitseliitlane, mina

### abstract_loc

### geo_loc
* kohanimed: Bristol, Sepphoris
* ehitised/äride füüsilised asukohad: pangamaja, multimeediastuudio, Kuku klubi, käisime arvutifirmas
* alad, mille geograafiline asukoht on defineeritav: põlengupaik, põhjapoolus, kaldapealne, tagaots, tolmupilv
* koju